Pose Estimation 
===============

Goal
----

In this section,
    -   We will learn to exploit 3d module to create some 3D effects in images.

Basics
------

This is going to be a small section. During the last session on camera calibration, you have found
the camera matrix, distortion coefficients etc. Given a pattern image, we can utilize the above
information to calculate its pose, or how the object is situated in space, like how it is rotated,
how it is displaced etc. For a planar object, we can assume Z=0, such that, the problem now becomes
how camera is placed in space to see our pattern image. So, if we know how the object lies in the
space, we can draw some 2D diagrams in it to simulate the 3D effect. Let's see how to do it.

Our problem is, we want to draw our 3D coordinate axis (X, Y, Z axes) on our chessboard's first
corner. X axis in blue color, Y axis in green color and Z axis in red color. So in-effect, Z axis
should feel like it is perpendicular to our chessboard plane.

First, let's load the camera matrix and distortion coefficients from the previous calibration
result.

In [ ]:
import glob

import cv2 as cv
import numpy as np

# termination criteria
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((6 * 7, 3), np.float32)
objp[:, :2] = np.mgrid[0:7, 0:6].T.reshape(-1, 2)

# Arrays to store object points and image points from all the images.
objpoints = []  # 3d point in real world space
imgpoints = []  # 2d points in image plane.

images = glob.glob("../py_calibration/images/left*.jpg")
first_image = cv.imread(images[0])

drawn = []  # store processed images for playback
found = 0

for fname in images:
    img = cv.imread(fname)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    # Find the chess board corners
    ret, corners = cv.findChessboardCorners(gray, (7, 6), None)
    # If found, add object points, image points (after refining them)
    if ret:
        objpoints.append(objp)
        corners2 = cv.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners)

ret, mtx, dist, rvecs, tvecs = cv.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)

Now let's create a function, draw which takes the corners in the chessboard (obtained using
**cv.findChessboardCorners()**) and **axis points** to draw a 3D axis.

In [ ]:
def draw(img, corners, imgpts):
    corner = tuple(corners[0].ravel().astype("int32"))
    imgpts = imgpts.astype("int32")
    img = cv.line(img, corner, tuple(imgpts[0].ravel()), (255, 0, 0), 5)
    img = cv.line(img, corner, tuple(imgpts[1].ravel()), (0, 255, 0), 5)
    img = cv.line(img, corner, tuple(imgpts[2].ravel()), (0, 0, 255), 5)
    return img

Then as in previous case, we create termination criteria, object points (3D points of corners in
chessboard) and axis points. Axis points are points in 3D space for drawing the axis. We draw axis
of length 3 (units will be in terms of chess square size since we calibrated based on that size). So
our X axis is drawn from (0,0,0) to (3,0,0), so for Y axis. For Z axis, it is drawn from (0,0,0) to
(0,0,-3). Negative denotes it is drawn towards the camera.

In [ ]:
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
objp = np.zeros((6 * 7, 3), np.float32)
objp[:, :2] = np.mgrid[0:7, 0:6].T.reshape(-1, 2)

axis = np.float32([[3, 0, 0], [0, 3, 0], [0, 0, -3]]).reshape(-1, 3)

Now, as usual, we load each image. Search for 7x6 grid. If found, we refine it with subcorner
pixels. Then to calculate the rotation and translation, we use the function,
**cv.solvePnPRansac()**. Once we those transformation matrices, we use them to project our **axis
points** to the image plane. In simple words, we find the points on image plane corresponding to
each of (3,0,0),(0,3,0),(0,0,3) in 3D space. Once we get them, we draw lines from the first corner
to each of these points using our generateImage() function. Done !!!

In [ ]:
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas

# Pre-load all images to avoid I/O during playback
loaded = [cv.imread(fname) for fname in images]
H, W = loaded[0].shape[:2]
canvas = Canvas(width=W, height=H)
frame_sl = widgets.IntSlider(value=0, min=0, max=len(loaded) - 1, description="Frame")
play_btn = widgets.Button(description="▶ Play", button_style="success")
playing = {"val": False}

# Cache for already-processed frames
_cache = {}


def _process_frame(idx):
    """Process a single frame on demand and cache the result."""
    if idx in _cache:
        return _cache[idx]
    img = loaded[idx].copy()
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    ret, corners = cv.findChessboardCorners(gray, (7, 6), None)
    if ret:
        corners2 = cv.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        ret, rvecs, tvecs = cv.solvePnP(objp, corners2, mtx, dist)
        imgpts, jac = cv.projectPoints(axis, rvecs, tvecs, mtx, dist)
        img = draw(img, corners2, imgpts)
    _cache[idx] = img
    return img


def _show_frame(idx):
    with hold_canvas():
        canvas.put_image_data(cv.cvtColor(_process_frame(idx), cv.COLOR_BGR2RGB), 0, 0)


def _on_slider(change):
    if not playing["val"]:
        _show_frame(change["new"])


def _on_play(_):
    import time

    if playing["val"]:
        playing["val"] = False
        play_btn.description = "▶ Play"
        play_btn.button_style = "success"
        return
    playing["val"] = True
    play_btn.description = "⏸ Pause"
    play_btn.button_style = "warning"
    for i in range(frame_sl.value, len(loaded)):
        if not playing["val"]:
            break
        frame_sl.value = i
        _show_frame(i)
        time.sleep(0.3)
    playing["val"] = False
    play_btn.description = "▶ Play"
    play_btn.button_style = "success"


frame_sl.observe(_on_slider, "value")
play_btn.on_click(_on_play)
_show_frame(0)
display(widgets.HBox([frame_sl, play_btn]), canvas)